# Phase 2 — Feature Engineering with SageMaker Processing Jobs

**What this notebook does:**
- Submits `src/preprocessing.py` as a SageMaker Processing Job
- Job runs on AWS 
- Outputs cleaned train/test CSVs to S3
- Validates processed data

## 1. Load config & session

In [ ]:
import boto3
from sagemaker.core.helper.session_helper import Session
import json
import pandas as pd
from io import StringIO

with open("../data/config.json") as f:
    cfg = json.load(f)

ROLE   = cfg["ROLE"]
BUCKET = cfg["BUCKET"]
REGION = cfg["REGION"]
PREFIX = cfg["PREFIX"]

boto_session = boto3.Session(region_name=REGION)
sagemaker_session = Session(boto_session=boto_session)
s3 = boto_session.client("s3")

print(f"Bucket : {BUCKET}")
print(f"Region : {REGION}")
print(f"Role   : {ROLE[:60]}...")

[05/29/26 13:53:47] INFO     Found credentials in shared credentials file: ~/.aws/credentials   credentials.py:1392

sagemaker.config INFO - Not applying SDK defaults from location: C:\ProgramData\sagemaker\sagemaker\config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: C:\Users\ashri\AppData\Local\sagemaker\sagemaker\config.yaml
Bucket : credit-risk-mlops-svp
Region : us-east-1
Role   : arn:aws:iam::064990711629:role/credit-risk...


## 2. Submit SageMaker Processing Job

In [ ]:
import sagemaker
from sagemaker.core.processing import ScriptProcessor  
from sagemaker.core.shapes import ProcessingInput, ProcessingOutput, ProcessingS3Input, ProcessingS3Output
from sagemaker.core import image_uris

input_data = f"s3://{BUCKET}/{PREFIX}/raw/application_train.csv"

processor = ScriptProcessor(
    image_uri=image_uris.retrieve(
        framework="sklearn",
        region=REGION,
        version="1.2-1",
        py_version="py3",
        instance_type="ml.t3.medium",
    ),
    instance_count=1,
    instance_type="ml.t3.medium",
    base_job_name="credit-data-preprocessing",
    role=ROLE,
    sagemaker_session=sagemaker_session
)

print("Submitting Processing job...")

processor.run(
    code = "../src/preprocessing.py",
    inputs = [
        ProcessingInput(
            input_name = "input-1",
            s3_input = ProcessingS3Input(
                s3_uri = input_data,
                local_path = "/opt/ml/processing/input",
                s3_data_type = "S3Prefix",
                s3_input_mode ="File",
                s3_data_distribution_type = "FullyReplicated",
            )
        )
    ],
    outputs = [
        ProcessingOutput(
            output_name = "train",
            s3_output = ProcessingS3Output(
                s3_uri = f"s3://{BUCKET}/{PREFIX}/processed/train/",
                local_path = "/opt/ml/processing/output/train",
                s3_upload_mode = "EndOfJob"
            )
        ),
      ProcessingOutput(
            output_name = "test",
            s3_output = ProcessingS3Output(
                s3_uri = f"s3://{BUCKET}/{PREFIX}/processed/test/",
                local_path = "/opt/ml/processing/output/test",
                s3_upload_mode = "EndOfJob"
            )
        ),  
    ],
    arguments = [
        "--input-data", "/opt/ml/processing/input",
        "--test-size", "0.2",
        "--random-state", "42"
    ],
    wait = True,
    logs = True
)

print("\nProcessing Job Completed")



Submitting Processing job...


[05/29/26 12:50:36] INFO     SageMaker Python SDK will collect telemetry to help us better telemetry_logging.py:110
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

[05/29/26 12:50:41] INFO     Creating processing-job with name                                    processing.py:614
                             credit-data-preprocessing-2026-05-29-07-20-36-280                                     

e:\Prashu\GitHub projects\Credit_Risk_AWS\senv\Lib\site-packages\rich\live.py:260: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

Output()

In [ ]:
processor.latest_job.refresh()
processor.latest_job.processing_job_status

## 3. Inspect job results

In [3]:
# List output files in S3
for prefix in [f"{PREFIX}/processed/train/", f"{PREFIX}/processed/test/"]:
    response = s3.list_objects_v2(Bucket=BUCKET, Prefix=prefix)
    for obj in response.get("Contents", []):
        size_mb = obj["Size"] / (1024 * 1024)
        print(f"s3://{BUCKET}/{obj['Key']}  ({size_mb:.1f} MB)")

s3://credit-risk-mlops-svp/credit-risk/processed/train/  (0.0 MB)
s3://credit-risk-mlops-svp/credit-risk/processed/train/train.csv  (93.4 MB)
s3://credit-risk-mlops-svp/credit-risk/processed/test/  (0.0 MB)
s3://credit-risk-mlops-svp/credit-risk/processed/test/test.csv  (23.3 MB)


In [4]:
# Preview processed training data from s3
obj = s3.get_object(Bucket=BUCKET, Key = f"{PREFIX}/processed/train/train.csv")
train_df = pd.read_csv(obj["Body"])

print(f"Train shape   : {train_df.shape}")
print(f"Target balance: {train_df['TARGET'].value_counts().to_dict()}")
print(f"\nNew engineered features:")
engineered = [c for c in train_df.columns if c in [
    "CREDIT_INCOME_RATIO", "ANNUITY_INCOME_RATIO", "CREDIT_TERM",
    "GOODS_CREDIT_RATIO", "AGE_YEARS", "YEARS_EMPLOYED",
    "EMPLOYMENT_AGE_RATIO", "DOCS_SUBMITTED"
]]
print(train_df[engineered].describe().round(3).to_string())


Train shape   : (246008, 81)
Target balance: {0: 226148, 1: 19860}

New engineered features:
       CREDIT_INCOME_RATIO  ANNUITY_INCOME_RATIO  CREDIT_TERM  GOODS_CREDIT_RATIO   AGE_YEARS  YEARS_EMPLOYED  EMPLOYMENT_AGE_RATIO  DOCS_SUBMITTED
count           246008.000            246008.000   246008.000          246008.000  246008.000      246008.000            246008.000      246008.000
mean                 3.959                 0.181        0.054               0.901      43.916           5.362                 0.126           0.930
std                  2.688                 0.095        0.022               0.097      11.953           6.325                 0.132           0.344
min                  0.005                 0.000        0.022               0.167      20.518           0.000                 0.000           0.000
25%                  2.019                 0.115        0.037               0.835      33.975           0.797                 0.021           1.000
50%                

In [5]:
# Confirm no nulls remain
null_count = train_df.isnull().sum().sum()
print(f"Remaining nulls in training data: {null_count}")
assert null_count == 0, "ERROR: Nulls found — preprocessing failed!"
print("Validation PASSED — data is clean and ready for training.")

Remaining nulls in training data: 0
Validation PASSED — data is clean and ready for training.


In [7]:
# Save processing job name for reference
import json

processing_job_name = "credit-data-preprocessing"
print(f"Processing job name: {processing_job_name}")

with open("../data/config.json") as f:
    cfg = json.load(f)

cfg["PROCESSING_JOB_NAME"] = processing_job_name
cfg["N_FEATURES"]          = train_df.shape[1] - 1  # exclude TARGET

with open("../data/config.json", "w") as f:
    json.dump(cfg, f, indent=2)

print(f"Config updated. Features: {cfg['N_FEATURES']}")

Processing job name: credit-data-preprocessing
Config updated. Features: 80
